In [ ]:
#@title Install Required Libraries
!pip install -q -U trl
!pip install -q -U bitsandbytes

In [ ]:
#@title SETUP

import datetime
import json
import os
import torch

from datasets import load_dataset
from peft import get_peft_model, LoraConfig, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    BitsAndBytesConfig
)
from trl import DPOTrainer, DPOConfig

In [ ]:
#@title 1. CONFIG

# =================
# experiment config
# =================
EXPERIMENT_CONFIG = {
    # Experiment identification
    'experiment_name': ###########, # with non-info ties and some noise

    'version': 'v1',
    'description': 'DPO training on strict P + ties non-informative + noise',

    # Training dataset
    'train_dataset': ##############,  # add from data generation

    'train_dataset_split': 'train',

    'train_samples': 5000,                              # None or 10000 for strong, 2000 for weak, and 5000 for medium

    # Model config
    'base_model': 'meta-llama/Llama-3.2-1B-Instruct',
    'lora_r': 16,
    'lora_alpha': 32,
    'lora_dropout': 0.05,

    # Training hyperparameters
    'learning_rate': 1e-5,                     # (maybe was too high before 5e-5)
    'num_epochs': 1,
    'per_device_batch_size': 4,
    'gradient_accumulation_steps': 2,
    'beta': 0.1,
    'max_prompt_length': 768,
    'max_length': 1024,

    # Training notes
    'notes': 'Mixed training with non-informative ties - alpha=0.9',

    'timestamp': datetime.datetime.now().strftime('%Y%m%d_%H%M%S'),
    'gpu_type': 'L4'                               # or 'L4', 'A100'
}

# ======================
# generate experiment ID
# ======================
EXPERIMENT_ID = f"{EXPERIMENT_CONFIG['experiment_name']}_{EXPERIMENT_CONFIG['version']}_{EXPERIMENT_CONFIG['timestamp']}"

print("="*80)
print(f"🔬 EXPERIMENT: {EXPERIMENT_ID}")
print("="*80)
print(f"Description: {EXPERIMENT_CONFIG['description']}")
print(f"Training on: {EXPERIMENT_CONFIG['train_dataset']}")
print(f"Samples: {EXPERIMENT_CONFIG['train_samples']}")
print(f"Learning Rate: {EXPERIMENT_CONFIG['learning_rate']}")
print(f"Beta: {EXPERIMENT_CONFIG['beta']}")
print("="*80)

In [ ]:
#@title 2. SET UP GOOGLE DRIVE
from google.colab import drive
drive.mount('/content/drive')

BASE_EXPERIMENTS_DIR = '/content/drive/MyDrive/dpo_experiments'
EXPERIMENT_DIR = f"{BASE_EXPERIMENTS_DIR}/{EXPERIMENT_ID}"
MODEL_SAVE_PATH = f"{EXPERIMENT_DIR}/model"

# ==================
# create directories
# ==================
os.makedirs(MODEL_SAVE_PATH, exist_ok=True)

# ==================================
# save experiment config immediately
# ==================================
with open(f"{EXPERIMENT_DIR}/config.json", 'w') as f:
    json.dump(EXPERIMENT_CONFIG, f, indent=2)

print(f"✅ Experiment directory: {EXPERIMENT_DIR}")

In [ ]:
#@title 3. LOAD DATASET

print("\n" + "="*80)
print("LOADING DATASET")
print("="*80)

# =========================
# load the training dataset
# =========================
dataset = load_dataset(
    EXPERIMENT_CONFIG['train_dataset'],
    split=EXPERIMENT_CONFIG['train_dataset_split']
)

print(f"Total dataset size: {len(dataset)}")

# ============================
# CRITICAL FIX: SHUFFLE FIRST!
# ============================
print("\n🔀 Shuffling dataset to prevent temporal ordering bias...")
dataset = dataset.shuffle(seed=42)

# Verify shuffle quality
print("\nVerifying shuffle quality...")
first_100 = dataset.select(range(100))
one_count_first = sum(1 for ex in first_100 if 'Option ONE' in ex['chosen'])
print(f"  First 100 examples: {one_count_first} chose ONE, {100-one_count_first} chose TWO")

if EXPERIMENT_CONFIG['train_samples']:
    middle_start = len(dataset) // 2
    middle_100 = dataset.select(range(middle_start, middle_start + 100))
    one_count_middle = sum(1 for ex in middle_100 if 'Option ONE' in ex['chosen'])
    print(f"  Middle 100 examples: {one_count_middle} chose ONE, {100-one_count_middle} chose TWO")

# ========================
# take subset if specified
# ========================
if EXPERIMENT_CONFIG['train_samples']:
    print(f"\n✂️  Taking subset of {EXPERIMENT_CONFIG['train_samples']} samples...")
    dataset = dataset.select(range(min(EXPERIMENT_CONFIG['train_samples'], len(dataset))))

    # Verify subset balance
    subset_one = sum(1 for ex in dataset if 'Option ONE' in ex['chosen'])
    subset_two = len(dataset) - subset_one
    print(f"  Subset balance: {subset_one} ONE ({subset_one/len(dataset)*100:.1f}%), "
          f"{subset_two} TWO ({subset_two/len(dataset)*100:.1f}%)")

    if abs(subset_one - subset_two) > len(dataset) * 0.1:
        print(f"  ⚠️  WARNING: Subset imbalance > 10%!")

# ====================
# split into train/val
# ====================
dataset = dataset.train_test_split(test_size=0.1, seed=42)

print(f"\n✅ Final split:")
print(f"  Training samples: {len(dataset['train'])}")
print(f"  Validation samples: {len(dataset['test'])}")

# Final verification on training set
train_one = sum(1 for ex in dataset['train'] if 'Option ONE' in ex['chosen'])
train_two = len(dataset['train']) - train_one
print(f"\n  Training set balance: {train_one} ONE ({train_one/len(dataset['train'])*100:.1f}%), "
      f"{train_two} TWO ({train_two/len(dataset['train'])*100:.1f}%)")

if abs(train_one - train_two) > len(dataset['train']) * 0.1:
    print(f"  🚨 ERROR: Training set is imbalanced! Do not proceed with training!")
else:
    print(f"  ✅ Training set is properly balanced")

print("="*80)

In [ ]:
#@title 4. MODEL CONFIGURATION

print("\n" + "="*80)
print("CONFIGURING MODEL")
print("="*80)

MODEL_NAME = EXPERIMENT_CONFIG['base_model']
OUTPUT_DIR = "./temp_training_output"

# =======================================
# LoRA configuration for memory efficient
# =======================================
lora_config = LoraConfig(
    r=EXPERIMENT_CONFIG['lora_r'],
    lora_alpha=EXPERIMENT_CONFIG['lora_alpha'],
    lora_dropout=EXPERIMENT_CONFIG['lora_dropout'],
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias="none",
    task_type="CAUSAL_LM",
)

# ===============================================
# 4-bit quantization config for memory efficiency
# ===============================================
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

In [ ]:
#@title 5. LOAD TOKENIZER AND MODEL

print("\nLoading tokenizer and model...")

# ==============
# load tokenizer
# ==============
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"                       # why this is important for DPO?
tokenizer.truncation_side = "left"

# ==================================
# load model with 4-bit quantization
# ==================================
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.bfloat16,
)

# ================================
# prepare model for k-bit training
# ================================
model = prepare_model_for_kbit_training(model)

# =================
# add LoRA adapters
# =================
model = get_peft_model(model, lora_config)

# ==========================
# print trainable parameters
# ==========================
model.print_trainable_parameters()

In [ ]:
#@title 6. LOAD REFERENCE MODEL (for DPO)

# ============================================================================================
# The reference model is used to compute KL divergence in DPO. We use the same quantized model
# ============================================================================================
ref_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.bfloat16,
)

In [ ]:
#@title 7. CONFIGURE DPO TRAINING

# ==================
# training arguments
# ==================
training_args = DPOConfig(
    output_dir=OUTPUT_DIR,

    # Training hyperparameters
    num_train_epochs=EXPERIMENT_CONFIG['num_epochs'],
    per_device_train_batch_size=EXPERIMENT_CONFIG['per_device_batch_size'],
    per_device_eval_batch_size=EXPERIMENT_CONFIG['per_device_batch_size'],
    gradient_accumulation_steps=EXPERIMENT_CONFIG['gradient_accumulation_steps'],
    gradient_checkpointing=True,

    # Optimizer settings
    learning_rate=EXPERIMENT_CONFIG['learning_rate'],
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    optim="paged_adamw_8bit",  # Memory-efficient optimizer

    # DPO specific
    beta=EXPERIMENT_CONFIG['beta'],                             # KL penalty coeff
    # max_prompt_length=EXPERIMENT_CONFIG['max_prompt_length'],
    max_length=EXPERIMENT_CONFIG['max_length'],
    # padding_value=tokenizer.pad_token_id,

    # Logging and saving
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,

    # Memory optimization
    bf16=True,
    dataloader_num_workers=2,  # Add parallel data loading
    dataloader_pin_memory=True,  # Speed up data transfer

    # Misc
    seed=42,
    report_to="none",  # Change to "wandb" if you want to log to W&B
)

In [ ]:
#@title 8. INITIALIZE DPO TRAINER

# ======================
# define the DPO trainer
# ======================
dpo_trainer = DPOTrainer(
    model=model,
    ref_model=ref_model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
)

In [ ]:
#@title TRAIN THE MODEL

print("\n" + "="*80)
print("🚀 STARTING TRAINING")
print("="*80)

# ===============
# train the model
# ===============
print("\nStarting DPO training...")
train_output = dpo_trainer.train()

print("\n✅ Training complete!")

# =====================
# save training history
# =====================
training_history = {
    'train_loss': train_output.training_loss if hasattr(train_output, 'training_loss') else None,
    'training_time': str(train_output.metrics.get('train_runtime', 'N/A')),
    'samples_per_second': train_output.metrics.get('train_samples_per_second', 'N/A'),
}

with open(f"{EXPERIMENT_DIR}/training_history.json", 'w') as f:
    json.dump(training_history, f, indent=2)

In [ ]:
#@title SAVE THE FINE-TUNED MODEL

print("\n" + "="*80)
print("💾 SAVING MODEL")
print("="*80)

# ====================
# save fine-tune model
# ====================
dpo_trainer.save_model(MODEL_SAVE_PATH)
tokenizer.save_pretrained(MODEL_SAVE_PATH)

print(f"✅ Model saved to: {MODEL_SAVE_PATH}")
print(f"Size: ~50MB (LoRA adapters)")

# ========================
# save additional metadata
# ========================
model_info = {
    'experiment_id': EXPERIMENT_ID,
    'base_model': EXPERIMENT_CONFIG['base_model'],
    'trained_on_dataset': EXPERIMENT_CONFIG['train_dataset'],
    'training_samples': len(dataset['train']),
    'lora_config': {
        'r': EXPERIMENT_CONFIG['lora_r'],
        'alpha': EXPERIMENT_CONFIG['lora_alpha'],
        'dropout': EXPERIMENT_CONFIG['lora_dropout']
    },
    'training_hyperparams': {
        'learning_rate': EXPERIMENT_CONFIG['learning_rate'],
        'beta': EXPERIMENT_CONFIG['beta'],
        'epochs': EXPERIMENT_CONFIG['num_epochs']
    },
    'timestamp': EXPERIMENT_CONFIG['timestamp']
}

with open(f"{MODEL_SAVE_PATH}/model_info.json", 'w') as f:
    json.dump(model_info, f, indent=2)